# 02 - AF3 setup for UPOs with heme and Mg

## Introduction

This notebook shows a practical workflow to run AlphaFold 3 (AF3) jobs for UPO sequences with one heme cofactor (`HEM`) and one magnesium ion (`MG`) using `prepare_proteins`.

The tutorial covers:

- Preparing the AF3 input JSON files with `setUpAlphaFold3`
- Submitting the generated commands
- Collecting and exporting top models with `collectAlphaFold3Results`
- Applying the built-in strict PROT-HEME-MG filter


## 1. Import modules

As in the existing project tutorials, we use both `prepare_proteins` and `bsc_calculations`.


In [ ]:
import json
import os
import shutil
import subprocess

import bsc_calculations
import pandas as pd
import prepare_proteins


## 2. Check runtime prerequisites

`setUpAlphaFold3` generates commands that expect:

- `bsc_alphafold` to be available in `PATH`
- `sbatch` to be available in `PATH`
- `WEIGHTS` to point to AF3 weights (or be exported by your SLURM helper)
- `bsc_calculations` importable in Python

Adjust these checks if your cluster environment resolves these tools differently.


In [ ]:
weights = os.environ.get("WEIGHTS")
if not weights:
    print("WARNING: WEIGHTS is not defined in this shell.")
else:
    print(f"WEIGHTS={weights}")

print("bsc_alphafold:", shutil.which("bsc_alphafold"))
print("sbatch:", shutil.which("sbatch"))
print("bsc_calculations module:", bsc_calculations.__file__)


## 3. Prepare a FASTA file with UPO sequences

Use your real UPO sequences in production. The short sequences below are placeholders so the setup code can run.


In [ ]:
fasta_path = "upo_sequences.fasta"

if not os.path.exists(fasta_path):
    with open(fasta_path, "w") as handle:
        handle.write(
            ">UPO_001\n"
            "MALVAVLLAASAFAAPVAAEGVTVVGPDATVKPASPVTVTTVGATPTVTAVDGYTVRV\n"
            ">UPO_002\n"
            "MKTLTALALALAGSAFAAPAQATVVGPKATVKPSAPVTVTTVGATPTVTVVDGYTVRV\n"
        )

print("FASTA file:", fasta_path)


## 4. Initialize `sequenceModels`

The FASTA headers become model names in the generated AF3 jobs.


In [ ]:
sequences = prepare_proteins.sequenceModels(fasta_path)
print("Loaded models:", sequences.sequences_names)


## 5. Generate AF3 jobs for protein + HEM + MG

Important: keep ligand order as `HEM`, then `MG` when you plan to use the built-in strict filter later.

The strict filter assumes chain indices:

- 0 -> protein
- 1 -> heme
- 2 -> magnesium


In [ ]:
job_folder = "af3_upo_heme_mg"
runner_name = "runner"

jobs = sequences.setUpAlphaFold3(
    job_folder=job_folder,
    ligands=["HEM", "MG"],
    model_seeds=[1, 2, 3, 4, 5],
    runner_name=runner_name,
    skip_finished=True,
    benchmark=True,
)

print(f"Prepared {len(jobs)} jobs")
if jobs:
    print("\nExample command:\n")
    print(jobs[0])


## 6. Inspect a generated AF3 input JSON

Each model gets `input/<MODEL>.json` with `dialect="alphafold3"` and the sequence+ligand blocks.


In [ ]:
first_model = sequences.sequences_names[0]
json_path = os.path.join(job_folder, first_model, "input", f"{first_model}.json")

with open(json_path) as handle:
    payload = json.load(handle)

print("JSON path:", json_path)
print(json.dumps(payload, indent=2))


## 7. Check runner script availability

`setUpAlphaFold3` generates commands that call `sbatch <runner_name> input` inside each model directory.

Make sure that runner script exists for every model folder.


In [ ]:
missing_runner = []
for model_name in sequences.sequences_names:
    candidate = os.path.join(job_folder, model_name, runner_name)
    if not os.path.exists(candidate):
        missing_runner.append(candidate)

if missing_runner:
    print("Missing runner files:")
    for path in missing_runner:
        print(" -", path)
else:
    print("Runner scripts detected for all models")


### Optional: write a minimal runner template

This is a simple placeholder and should be adapted to your SLURM partition, account, and runtime policy.


In [ ]:
write_template_runner = False

runner_template = """#!/bin/bash
#SBATCH --job-name=af3_batch
#SBATCH --output=af3_%j.out
#SBATCH --error=af3_%j.err
#SBATCH --time=24:00:00
#SBATCH --cpus-per-task=8
#SBATCH --mem=32G

set -euo pipefail
INPUT_DIR=${1:-input}
echo "Runner invoked for $INPUT_DIR"
"""

if write_template_runner:
    for model_name in sequences.sequences_names:
        runner_path = os.path.join(job_folder, model_name, runner_name)
        with open(runner_path, "w") as handle:
            handle.write(runner_template)
        os.chmod(runner_path, 0o755)
    print("Runner templates written")
else:
    print("Skipped writing runner templates (set write_template_runner=True to enable)")


## 8. Create a SLURM array script with `bsc_calculations`

Following the style of tutorial 01, we build a SLURM array script with `bsc_calculations.mn5.jobArrays`.

For AF3 on MN5, `program="alphafold3"` enables AF3-specific setup in the helper.

Useful flags in this call:

- `partition`: queue (`acc_bscls`, `acc_debug`, `gp_bscls`, `gp_debug`)
- `gpus`: number of GPUs requested
- `exports`: optional environment exports (for example custom `WEIGHTS`)
- `jobs_range` / `group_jobs_by`: split large command lists into manageable arrays


In [ ]:
slurm_script = "slurm_array_af3_upo.sh"
slurm_job_name = "AF3_UPO_HEM_MG"

# Recommended MN5 partition for AF3
partition = "acc_bscls"

# Optional: override WEIGHTS in the generated script
exports = [f"WEIGHTS={weights}"] if weights else None

bsc_calculations.mn5.jobArrays(
    jobs,
    script_name=slurm_script,
    job_name=slurm_job_name,
    output=slurm_job_name,
    partition=partition,
    program="alphafold3",
    gpus=1,
    exports=exports,
)

print(f"Generated {slurm_script}")
print(f"Submit with: sbatch {slurm_script}")

launch_slurm = False
if launch_slurm:
    subprocess.run(["sbatch", slurm_script], check=True)
else:
    print("Dry run only. Set launch_slurm=True to submit.")


## 9. Collect AF3 ranking scores and export top models

After jobs finish, collect metrics and export top predictions as PDB files.

Use `append_model_index=True` when exporting more than one top model per sequence.


In [ ]:
scores_df, selected_df, copied_paths = sequences.collectAlphaFold3Results(
    af_folder=job_folder,
    output_folder="af3_upo_heme_mg_top_models",
    metric="ranking_score",
    top_models=5,
    append_model_index=True,
    return_selected=True,
)

print(scores_df.head())
print(selected_df.head())
print(copied_paths)


## 10. Apply strict PROT-HEME-MG filtering (optional)

`filter_strict=True` adds model-level pass/fail annotations with reasons.


In [ ]:
filtered_scores, filtered_selected, filtered_copied, filter_info = sequences.collectAlphaFold3Results(
    af_folder=job_folder,
    output_folder="af3_upo_heme_mg_filtered_models",
    metric="ranking_score",
    top_models=5,
    append_model_index=True,
    return_selected=True,
    filter_strict=True,
    return_filter=True,
)

print(filter_info["counts"])
print(filter_info["reasons"])


In [ ]:
if "filtered_scores" in globals():
    cols = [
        "ranking_score",
        "summary_iptm",
        "prot_ptm",
        "pair_iptm_prot_heme",
        "pair_iptm_prot_mg",
        "pae_min_prot_heme",
        "pae_min_prot_mg",
        "af3_strict_pass",
        "af3_strict_reason",
    ]
    cols = [c for c in cols if c in filtered_scores.columns]
    print(filtered_scores[cols].head())


## 11. Per-model ligand customization (optional)

You can provide ligand specifications per model while keeping a default for the rest.


In [ ]:
custom_jobs = sequences.setUpAlphaFold3(
    job_folder="af3_upo_heme_mg_custom",
    model_seeds=[1, 2, 3],
    ligands={
        "default": ["HEM", "MG"],
        "UPO_007": {"HEM": 1, "MG": 2},
    },
    skip_finished=True,
)

print("Custom jobs:", len(custom_jobs))


## 12. Troubleshooting checklist

- If no `ranking_scores.csv` files exist, AF3 likely failed before finishing.
- If `metric` is not found, inspect available columns in the generated ranking CSV.
- If copied structures are missing, check `overwrite` and `append_model_index` options.
- If strict filtering gives unexpected results, verify ligand order is still protein -> heme -> magnesium.
- If jobs fail immediately, validate `WEIGHTS`, `bsc_alphafold`, and `sbatch` in your active environment.
- If SLURM submission fails, open the generated `slurm_array_af3_upo.sh` and verify partition/account values for your project.
